# video-base-imagem-padrao.ipynb — Narrated Video (image background, standard)

> 🖥️ **CPU basta** — este notebook não usa GPU. Deixe o acelerador em *Nenhum*: não muda nada aqui e poupa sua cota de GPU, que é limitada.

Generates the **video base** using Pixabay PHOTOS (still images) as
background, **standard mode**: narration + credited/logoed photos (random
unused rows, DURACAO_CLIPE seconds each, default 5s) + background music.
No verse matching — for that, use `video-base-imagem-versiculo.ipynb`. No
subtitles yet either — that's what the next notebooks are for.

For a VIDEO-clip background instead, use `video-base-video-padrao.ipynb`.

**How to use:**
1. Run cells top to bottom.
2. Edit only the "⚙️ Configuration" cell to start a new video.
3. Each step skips automatically what's already done (checkpoint).


In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP — packages, Google Drive, and modules (run once per session) ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── System packages ──────────────────────────────────────────────────────────
!apt-get -qq -y install ffmpeg espeak-ng > /dev/null 2>&1
print('✅ ffmpeg + espeak-ng')

# ── Python packages ───────────────────────────────────────────────────────────
!pip install -q edge-tts pandas gdown yt-dlp nest_asyncio gspread
print('✅ Python packages')

# ── Mount Drive (unmount first to avoid a stuck session) ────────────────────
from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)
print('✅ Drive mounted')

# ── Copy modules from Drive to /content/pipeline ────────────────────────────
import shutil, os, sys, logging
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"  # fixed for the whole project (same value as Configuration)
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos")
DESTINO = Path("/content/pipeline")

if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} modules copied from {PASTA_MODULOS}")

    # ── A cópia trouxe TODOS os módulos? ───────────────────────────────────────
    # "N módulos copiados" sozinho não quer dizer nada. E o modo de falhar aqui é
    # traiçoeiro: o Drive montado do Colab popula a listagem da pasta com atraso,
    # então um copytree logo depois do mount às vezes enxerga só parte dos
    # arquivos. Já aconteceu de copiar 13 de 31 -- com visto verde -- e o notebook
    # quebrar muito depois, num import, longe da causa.
    #
    # A conferência é de três pontas, porque a causa muda o conserto:
    #   manifesto  o que o repositório tem  (versionado; chega pela cópia)
    #   Drive      o que chegou lá
    #   VM         o que a cópia desta célula trouxe
    # A conferência tem duas perguntas, e SÓ UMA delas precisa do manifesto:
    #
    #   Drive → VM   a cópia acima trouxe tudo?      dá pra ver aqui mesmo
    #   repo → Drive o Drive está em dia?            só o manifesto sabe
    #
    # A versão anterior amarrava as duas ao manifesto: sem ele, imprimia um
    # aviso e seguia SEM CONFERIR NADA. Foi assim que "✅ 13 modules copied"
    # passou com visto verde num Drive que tinha 31 -- justamente no dia em
    # que o manifesto ainda não existia. Comparar 13 com 31 nunca dependeu de
    # manifesto nenhum.
    _no_drive = {f.name for f in PASTA_MODULOS.glob("*.py")}
    _na_vm    = {f.name for f in DESTINO.glob("*.py")}

    # ── Drive → VM ────────────────────────────────────────────────────────
    # O Drive montado do Colab popula a listagem da pasta com atraso, então um
    # copytree logo depois do mount às vezes enxerga só parte dos arquivos.
    # Uma segunda passada, com o mount já quente, costuma resolver.
    _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"   ⏳ {len(_nao_copiados)} módulo(s) não vieram na 1ª passada — copiando de novo")
        for _n in _nao_copiados:
            shutil.copyfile(PASTA_MODULOS / _n, DESTINO / _n)
        _na_vm = {f.name for f in DESTINO.glob("*.py")}
        _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"\n🚨 {len(_nao_copiados)} módulo(s) estão no Drive mas não copiaram:")
        for _n in _nao_copiados:
            print(f"     {_n}")
        raise SystemExit("Rode ESTA célula de novo — o Drive montado ainda estava acordando.")
    print(f"   ✅ os {len(_no_drive)} módulos do Drive chegaram na VM")

    # ── repositório → Drive ───────────────────────────────────────────────
    _manifesto = PASTA_MODULOS / "_manifesto.txt"
    if not _manifesto.exists():
        print("   ⚠️  sem _manifesto.txt: não dá pra saber se o DRIVE está atrás")
        print("      do repositório. Ele é versionado — rode o repositorio-sincronizar.")
    else:
        _esperados = {l.strip() for l in _manifesto.read_text().splitlines()
                      if l.strip() and not l.startswith("#")}
        _fora_do_drive = sorted(_esperados - _no_drive)
        if _fora_do_drive:
            print(f"\n🚨 {len(_fora_do_drive)} módulo(s) não estão no DRIVE:")
            for _n in _fora_do_drive:
                print(f"     {_n}")
            raise SystemExit("Rode o repositorio-sincronizar.ipynb — o Drive está atrás do repositório.")
        print(f"   ✅ e batem com os {len(_esperados)} do manifesto")

    # ── O Python está segurando a versão anterior? ────────────────────────
    # Copiar arquivo novo por cima não desfaz um import já feito: o Python
    # guarda o módulo em sys.modules e reaproveita. Numa sessão longa, isso
    # faz o notebook rodar com o config.py de ontem mesmo depois de um sync
    # perfeito -- e o sintoma aparece longe da causa (nome de arquivo que
    # mudou, padrão que era pra ter mudado e não mudou). Descarregar aqui
    # equivale a reiniciar o runtime, sem perder o resto da sessão.
    _recarregar = [_n for _n, _m in list(sys.modules.items())
                   if getattr(_m, "__file__", None) and str(DESTINO) in str(_m.__file__)]
    for _n in _recarregar:
        del sys.modules[_n]
    if _recarregar:
        print(f"   ♻️  {len(_recarregar)} módulo(s) já importados foram descarregados —")
        print(f"      o import vai reler a cópia nova (rode as células seguintes de novo)")
else:
    print(f"❌ Modules folder not found: {PASTA_MODULOS}")
    print("   Make sure the .py files are in pipeline/modulos/ on Drive.")

if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

# ── O ambiente combina com o que este notebook faz? ────────────────────────
# Cota de GPU do Colab é limitada e some sem aviso -- e parte da nossa foi
# gasta em notebook que não usa GPU pra nada, rodando com GPU só porque a
# seleção ficou de antes. Silencioso quando combina.
try:
    from ambiente import avisar_gpu
    avisar_gpu(precisa=False)
except Exception:
    pass

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(name)-18s  %(levelname)s  %(message)s', datefmt='%H:%M:%S')
os.chdir('/content')
print('✅ Setup complete!')


✅ ffmpeg + espeak-ng
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 74.6 MB/s eta 0:00:00
✅ Python packages
Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
✅ Drive mounted
✅ 13 modules copied from /content/drive/MyDrive/narrated_video/pipeline/modulos
✅ Setup complete!


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURATION                                               ║
# ║  ✏️  Edit only this cell when starting a new video               ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── 1. VIDEO IDENTITY ──────────────────────────────────────────────────────
NOME_ORACAO = "40_Matt_02"           # short identifier, no spaces/accents
                                    # e.g. "40_matt_02", "genesis_01"

# ── 2. FULL TEXT — ÚLTIMO recurso, não o primeiro.
# A célula seguinte procura primeiro o roteiro deste vídeo no Drive, depois
# o web-biblia.json (capítulo bíblico sai de lá, conferido). Este texto só
# é usado quando nenhum dos dois responde.
TEXTO_ORACAO = (
"Now when Jesus was born in Bethlehem of Judea, in the days of King Herod. "
"Behold, wise men from the east came to Jerusalem. "
"Saying, Where is He who is born King of the Jews? "
"For we saw His star in the east and have come to worship Him. "
"When King Herod heard it, he was troubled. "
"And all Jerusalem with him. "
"Gathering together all the chief priests and scribes of the people. "
"He asked them where the Christ would be born. "
"They said to him: In Bethlehem of Judea, for thus it is written through the prophet. "
"You, Bethlehem, land of Judah, are in no way least among the princes of Judah. "
"For out of you shall come forth a governor who shall shepherd My people Israel. "
"Then Herod secretly called the wise men. "
"And learned from them exactly what time the star appeared. "
"He sent them to Bethlehem and said: Go and search diligently for the young child. "
"When you have found him, bring me word, so that I also may come and worship him. "
"They, having heard the King, went their way. "
"And behold, the star which they saw in the east went before them. "
"Until it came and stood over where the young child was. "
"When they saw the star, they rejoiced with exceedingly great joy. "
"They came into the house and saw the young child with Mary, his mother. "
"And they fell down and worshiped him. "
"Opening their treasures, they offered to him gifts: gold, frankincense, and myrrh. "
"Being warned in a dream not to return to Herod. "
"They went back to their own country another way. "
"Now when they had departed, behold, an angel of the Lord appeared to Joseph in a dream. "
"Saying, Arise and take the young child and his mother. "
"And flee into Egypt and stay there until I tell you. "
"For Herod will seek the young child to destroy him. "
"He arose and took the young child and his mother by night. "
"And departed into Egypt. "
"And was there until the death of Herod. "
"That it might be fulfilled which was spoken by the Lord through the prophet. "
"Out of Egypt I called My Son. "
"Then Herod, when he saw that he was mocked by the wise men, was exceedingly angry. "
"And sent out and killed all the male children who were in Bethlehem. "
"And in all the surrounding countryside, from two years old and under. "
"According to the exact time which he had learned from the wise men. "
"Then that which was spoken by Jeremiah the prophet was fulfilled. "
"Saying, A voice was heard in Rama, lamentation, weeping, and great mourning. "
"Rachel weeping for her children, and she would not be comforted. "
"Because they are no more. "
"But when Herod was dead, behold, an angel of the Lord appeared in a dream to Joseph in Egypt. "
"Saying, Arise and take the young child and his mother. "
"And go into the land of Israel. "
"For those who sought the young child's life are dead. "
"He arose and took the young child and his mother. "
"And came into the land of Israel. "
"But when he heard that Archelaus was reigning over Judea in the place of his father Herod. "
"He was afraid to go there. "
"Being warned in a dream, he withdrew into the region of Galilee. "
"And came and lived in a city called Nazareth. "
"That it might be fulfilled which was spoken through the prophets. "
"- He will be called a Nazarene! "
)

# ── 3. NARRATOR VOICE ──────────────────────────────────────────────────────
# ⚠️  Must match the LANGUAGE of TEXTO_ORACAO above (Edge TTS reads the text
#    with that voice's pronunciation — wrong voice = strange accent).
#
#    🇧🇷 Portuguese:  "pt-BR-AntonioNeural" (m) · "pt-BR-FranciscaNeural" (f)
#    🇺🇸 English:      "en-US-GuyNeural" (m) · "en-US-JennyNeural" (f)
#                      "en-US-ChristopherNeural" (m, deep/narrator)
#    🇪🇸 Spanish:      "es-ES-AlvaroNeural" (m) · "es-ES-ElviraNeural" (f)
#    🇫🇷 French:       "fr-FR-HenriNeural" (m) · "fr-FR-DeniseNeural" (f)
#    🇨🇳 Chinese:      "zh-CN-YunxiNeural" (m) · "zh-CN-XiaoxiaoNeural" (f)
#
VOZ_EDGE = "en-US-GuyNeural"

# ── 3b. MASTER LANGUAGE ────────────────────────────────────────────────────
# Language of TEXTO_ORACAO/VOZ_EDGE above -- must match. Used to find the
# right Whisper SRT later (caption-single-generate.ipynb).
IDIOMA_MESTRE = "en"

# ── 4. VOLUME — narration vs. background music ratio ─────────────────────────
VOLUME_NARRACAO = 1.0    # narration volume (1.0 = original, unchanged)
VOLUME_MUSICA   = 0.25   # background music volume, relative to narration
                          # (0.25 = music at 1/4 of narration'''s volume)

# ── 5. NARRATION SPEED ────────────────────────────────────────────────────
VELOCIDADE_AUDIO = 1.0   # 1.0 = original speed; 0.9 = 10% slower; 1.1 = 10% faster

# ── 6. IMAGE SHEET (Pixabay photos) ────────────────────────────────────────
# Google Sheet ID from its URL
# (https://docs.google.com/spreadsheets/d/THIS_PART_HERE/edit...). Must have
# "Imagem" (direct image URL), "Autor", and a status column matching
# NOME_COLUNA_STATUS_PLANILHA below.
ID_PLANILHA_IMAGENS_DRIVE = "1P2LydKeeoU5MsAPNl1qhno5qsD1q5BbMOeTbblOVU1E"  # pixabay-image-stock
NOME_COLUNA_STATUS_PLANILHA = "Downloading Ok"

# Each image is shown still for this many seconds before switching to the
# next one (sequential -- next UNUSED row in the sheet, same anti-repeat
# logic as the video mode).
DURACAO_CLIPE = 5

# ── 7. DRIVE ROOT FOLDER ──────────────────────────────────────────────────
PASTA_DRIVE_RAIZ = "narrated_video"     # ⚠️ DO NOT CHANGE — fixed for the whole project


# ── CHECK ──────────────────────────────────────────────────────────────────
print("=" * 60)
print("⚙️  CONFIGURATION (image background)")
print("=" * 60)
print(f"   Name:            {NOME_ORACAO}")
print(f"   Voice:           {VOZ_EDGE}")
print(f"   Master language:  {IDIOMA_MESTRE}")
print(f"   Narration volume:  {VOLUME_NARRACAO}")
print(f"   Music volume:      {VOLUME_MUSICA}")
print(f"   Speed:           {VELOCIDADE_AUDIO}x")
print(f"   Image sheet:     {ID_PLANILHA_IMAGENS_DRIVE or '⚠️ NOT SET'}")
print(f"   Seconds/image:   {DURACAO_CLIPE}")
print(f"   Drive root:      {PASTA_DRIVE_RAIZ}")
print(f"   Text:            {TEXTO_ORACAO[:60]}...")
print("=" * 60)
print("✅ Configuration ready — proceed to Script/Audio (optional) and Initialization")


⚙️  CONFIGURATION
   Name:            40_Matt_02
   Voice:           en-US-GuyNeural
   Narration volume:  1.0
   Music volume:      0.25
   Speed:           1.0x
   Background:      imagem (no sheet ID set!)
   Drive root:      narrated_video
   Text:            Now when Jesus was born in Bethlehem of Judea, in the days o...
✅ Configuration ready — proceed to Script/Audio (optional) and Initialization


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📄 TEXTO E ÁUDIO — o sistema busca, você só completa o que faltar ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# A ordem é de fora pra dentro: o que o sistema JÁ TEM ganha do que está
# digitado na Configuração. O texto da célula é o último recurso, não o
# primeiro -- ele fica lá parado de um vídeo pro outro, e um texto colado há
# meses vencendo a Bíblia conferida é o tipo de erro que não dá aviso nenhum.

from pathlib import Path
import shutil, subprocess, sys

if '/content/pipeline' not in sys.path:
    sys.path.insert(0, '/content/pipeline')

BASE = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}")
PASTA_VIDEO_DRIVE = BASE / "videos" / NOME_ORACAO
PASTA_VIDEO_DRIVE.mkdir(parents=True, exist_ok=True)

# ── TEXTO ───────────────────────────────────────────────────────────────────
#   1. <nome>_roteiro.txt na pasta do vídeo  -- edição sua PARA ESTE vídeo
#   2. dados_lexico/web-biblia.json          -- a WEB inteira, conferida
#   3. TEXTO_ORACAO da Configuração          -- último recurso
roteiro_drive = PASTA_VIDEO_DRIVE / f"{NOME_ORACAO}_roteiro.txt"
CAMINHO_BIBLIA = BASE / "pipeline" / "dados_lexico" / "web-biblia.json"

if roteiro_drive.exists():
    TEXTO_ORACAO = roteiro_drive.read_text(encoding="utf-8").strip()
    print(f"📄 Texto: {roteiro_drive.name} (roteiro deste vídeo, {len(TEXTO_ORACAO)} caracteres)")
else:
    _do_json, _motivo = None, ""
    if CAMINHO_BIBLIA.exists():
        # Duas coisas podem dar errado aqui, e elas pedem mensagens diferentes:
        # o nome não ser de capítulo bíblico ("oracao_bom_dia"), ou o capítulo
        # não estar no arquivo. Culpar o nome pelos dois mandaria você conferir
        # o lugar errado -- que é o defeito que essa mensagem existe pra evitar.
        import biblia_livros as _bl, biblia_texto as _bt
        _livro = _cap = None
        try:
            _livro, _cap = _bl.de_nome_projeto(NOME_ORACAO)
        except Exception:
            _motivo = f"{NOME_ORACAO} não é nome de capítulo bíblico"
        if _livro is not None:
            try:
                _do_json = _bt.roteiro_do_capitulo(NOME_ORACAO, CAMINHO_BIBLIA)
            except Exception as _e:
                _motivo = f"{_livro.nome} {_cap} não saiu do web-biblia.json — {_e}"

    if _do_json:
        # DOIS arquivos, com formatos diferentes de propósito:
        #
        #   <nome>_roteiro_versiculos.txt   "1 Now when Jesus... 2 Where is he..."
        #       o número é dado -- é por ele que o match casa cena com versículo
        #       e que a legenda sabe qual referência mostrar. É o que os
        #       notebooks de versículo e de queima procuram.
        #
        #   <nome>_roteiro.txt              "Now when Jesus... Where is he..."
        #       o número é lixo -- este vira TEXTO_ORACAO, e o Edge TTS leria
        #       "um. Agora quando Jesus... dois." em voz alta.
        #
        # Gravar um no lugar do outro não dá erro: dá uma narração contando
        # números, ou um match que não acha versículo nenhum.
        TEXTO_ORACAO = _bt.narracao_do_capitulo(NOME_ORACAO, CAMINHO_BIBLIA)
        print(f"📄 Texto: web-biblia.json — {_livro.nome} {_cap} ({len(TEXTO_ORACAO)} caracteres)")

        roteiro_drive.write_text(TEXTO_ORACAO, encoding="utf-8")
        print(f"   ↳ {roteiro_drive.name} (narração, sem números) — edite ali se quiser mudar")

        versiculos_drive = PASTA_VIDEO_DRIVE / f"{NOME_ORACAO}_roteiro_versiculos.txt"
        versiculos_drive.write_text(_do_json, encoding="utf-8")
        print(f"   ↳ {versiculos_drive.name} (com números) — usado pelo match e pelas legendas")
    else:
        print(f"📄 Texto: TEXTO_ORACAO da Configuração ({len(TEXTO_ORACAO)} caracteres)")
        if not CAMINHO_BIBLIA.exists():
            print("   (web-biblia.json não existe — rode o biblia-texto-baixar uma vez)")
        else:
            print(f"   ({_motivo})")

# ── ÁUDIO ───────────────────────────────────────────────────────────────────
#   1. pasta do vídeo        -- gravação própria, ou o que você subiu
#   2. assets/biblia_audio/  -- o estoque do biblia-audio-baixar (1189 caps)
#   3. nada aqui             -- o Edge TTS gera na fase da narração
# A busca é a mesma de video_pipeline.gerar_audio(); esta célula só adianta a
# cópia pra VM e te mostra o que vai ser usado ANTES de rodar o resto.
audio_local = Path(f"/content/{NOME_ORACAO}_audio.wav")
EXTENSOES_AUDIO = [".wav", ".mp3", ".m4a", ".ogg", ".flac"]

audio_drive, rotulo_audio = None, ""
for _pasta, _rotulo in ((PASTA_VIDEO_DRIVE, "pasta do vídeo"),
                        (BASE / "assets" / "biblia_audio", "estoque da Bíblia")):
    for _base in (f"{NOME_ORACAO}_audio", NOME_ORACAO):
        for _ext in EXTENSOES_AUDIO:
            _c = _pasta / f"{_base}{_ext}"
            if _c.exists():
                audio_drive, rotulo_audio = _c, _rotulo
                break
        if audio_drive: break
    if audio_drive: break

if audio_drive is None:
    print(f"\n🔊 Áudio: nenhum encontrado — o Edge TTS vai gerar com {VOZ_EDGE}")
    print("   (pra usar a gravação do David Williams, rode o biblia-audio-baixar)")
elif audio_local.exists():
    print(f"\n🔊 Áudio: {audio_local.name} já está nesta sessão ({audio_local.stat().st_size/1_048_576:.1f} MB)")
elif audio_drive.suffix == ".wav":
    shutil.copy2(audio_drive, audio_local)
    print(f"\n🔊 Áudio: {audio_drive.name} — {rotulo_audio} ({audio_local.stat().st_size/1_048_576:.1f} MB)")
else:
    print(f"\n🔊 Áudio: {audio_drive.name} — {rotulo_audio}, convertendo {audio_drive.suffix} → .wav ...")
    r = subprocess.run(
        ["ffmpeg", "-y", "-i", str(audio_drive), "-vn", "-acodec", "pcm_s16le",
         "-ar", "44100", str(audio_local)],
        capture_output=True, text=True,
    )
    if audio_local.exists():
        print(f"   ✅ {audio_local.name} ({audio_local.stat().st_size/1_048_576:.1f} MB)")
    else:
        print("   ❌ conversão falhou:")
        print(r.stderr[-800:])


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎙️ ÁUDIO DUBLADO DO YOUTUBE (opcional, quase sempre desnecessário) ║
# ║                                                                    ║
# ║  Só serve quando NÃO há áudio nem na pasta do vídeo nem no        ║
# ║  estoque assets/biblia_audio/. Com o estoque baixado (os 1189     ║
# ║  capítulos), esta célula não tem o que fazer e diz isso.          ║
# ╚══════════════════════════════════════════════════════════════════╝

import subprocess
from pathlib import Path

URL_DUBLAGEM    = "https://www.youtube.com/watch?v=4vTN7tBG3a8"  # vídeo com dublagem automática
IDIOMA_DUBLAGEM = "en"    # código do idioma da faixa que você quer
FORMAT_ID_MANUAL = ""     # em branco = automático. Se falhar, veja a lista e cole o ID exato.
LISTAR_TUDO     = False   # True = despeja TODAS as faixas (são ~150 linhas; já travou navegador)

audio_local = Path(f"/content/{NOME_ORACAO}_audio.wav")

# ── Já tem áudio? Então não baixa nada ──────────────────────────────────────
# Esta célula existia de quando o áudio precisava vir de algum lugar. Depois do
# biblia-audio-baixar, o capítulo está no estoque e o pipeline acha sozinho --
# baixar do YouTube por cima seria trocar a gravação da fonte por uma faixa
# recomprimida, e ainda depender do vídeo continuar no ar.
BASE = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}")
_ja_tem = None
if audio_local.exists():
    _ja_tem = f"{audio_local} (nesta sessão)"
else:
    for _pasta, _rotulo in ((BASE / "videos" / NOME_ORACAO, "pasta do vídeo"),
                            (BASE / "assets" / "biblia_audio", "estoque da Bíblia")):
        for _base in (f"{NOME_ORACAO}_audio", NOME_ORACAO):
            for _ext in (".wav", ".mp3", ".m4a", ".ogg", ".flac"):
                if (_pasta / f"{_base}{_ext}").exists():
                    _ja_tem = f"{_pasta / (_base + _ext)} ({_rotulo})"
                    break
            if _ja_tem: break
        if _ja_tem: break

if _ja_tem:
    print(f"⏭️  Já existe áudio — esta célula não precisa rodar.")
    print(f"    {_ja_tem}")
    print(f"    Se quiser MESMO a faixa do YouTube, apague o arquivo acima e rode de novo.")
else:
    # ── Listagem das faixas ─────────────────────────────────────────────────
    # Por padrão só os idiomas, uma linha. `yt-dlp -F` num vídeo com dublagem
    # automática cospe ~150 linhas (mesmo idioma em 7 formatos), e foi isso que
    # travou o navegador. Ligue LISTAR_TUDO só se precisar do ID exato.
    lista = subprocess.run(["yt-dlp", "-F", URL_DUBLAGEM], capture_output=True, text=True)
    linhas_audio = [l for l in lista.stdout.splitlines() if "audio only" in l]

    if LISTAR_TUDO:
        print(f"📋 {len(linhas_audio)} faixas de áudio:")
        for l in linhas_audio:
            print("  ", l)
    else:
        import re
        idiomas = sorted({m.group(1) for l in linhas_audio if (m := re.search(r"\[([\w-]+)\]", l))})
        print(f"📋 {len(linhas_audio)} faixas em {len(idiomas)} idioma(s): {', '.join(idiomas)}")
        print("   (LISTAR_TUDO = True pra ver todas — são muitas linhas)")

    formato = FORMAT_ID_MANUAL.strip() or f"ba[language^={IDIOMA_DUBLAGEM}]/bestaudio[language^={IDIOMA_DUBLAGEM}]"
    print(f"\n🎙️  Baixando (formato: {formato})...")

    resultado = subprocess.run(
        ["yt-dlp", "-f", formato, "--extract-audio", "--audio-format", "wav",
         "-o", "temp_dublagem.%(ext)s", URL_DUBLAGEM],
        capture_output=True, text=True,
    )

    temp_wav = Path("temp_dublagem.wav")
    if temp_wav.exists():
        temp_wav.replace(audio_local)
        print(f"✅ Áudio salvo: {audio_local.name} ({audio_local.stat().st_size/1_048_576:.2f} MB)")
    else:
        print("❌ Não achei uma faixa correspondente automaticamente.")
        print("   Ponha LISTAR_TUDO = True, rode de novo, copie o ID da faixa que quer")
        print("   (coluna da esquerda, ex: '233-1') e cole em FORMAT_ID_MANUAL. Detalhe do erro:")
        print(resultado.stderr[-800:])


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 INITIALIZE PIPELINE                                          ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
import nest_asyncio
nest_asyncio.apply()

from pathlib import Path

if '/content/pipeline' not in sys.path:
    sys.path.insert(0, '/content/pipeline')

from config import PipelineConfig
from video_pipeline import VideoPipeline
from checkpoint import Checkpoint

config = PipelineConfig(
    NOME_ORACAO                  = NOME_ORACAO,
    PASTA_DRIVE_RAIZ              = PASTA_DRIVE_RAIZ,
    TEXTO_ORACAO                  = TEXTO_ORACAO,
    VOZ_EDGE                      = VOZ_EDGE,
    IDIOMA_MESTRE                  = IDIOMA_MESTRE,
    VOLUME_NARRACAO               = VOLUME_NARRACAO,
    VOLUME_MUSICA                 = VOLUME_MUSICA,
    VELOCIDADE_AUDIO              = VELOCIDADE_AUDIO,
    MODO_ROTEIRO                  = "padrao",
    MODO_CLIPE                    = "imagem",
    ID_PLANILHA_IMAGENS_DRIVE     = ID_PLANILHA_IMAGENS_DRIVE,
    NOME_COLUNA_STATUS_PLANILHA   = NOME_COLUNA_STATUS_PLANILHA,
    DURACAO_CLIPE                  = DURACAO_CLIPE,
)

pipeline = VideoPipeline(config)
cp       = Checkpoint(nome_oracao=config.NOME_ORACAO)  # isolated per video — see checkpoint.py

print("=" * 60)
print("✅ PIPELINE INITIALIZED")
print("=" * 60)
print(f"   Video:       {config.NOME_ORACAO}")
print(f"   Folder:      {config.pasta_oracao}")
print(f"   Checkpoint:  {cp.proxima_fase_pendente() or 'all done'}")
print("=" * 60)
print(config.resumo())


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎤 NARRATION — generate audio with Edge TTS                     ║
# ║  Skips generation if the audio already exists (Drive/manual/etc.) ║
# ╚══════════════════════════════════════════════════════════════════╝

audio = pipeline.gerar_audio()
print(f'✅ {audio}  ({audio.stat().st_size/1024:.0f} KB)')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🖼️🏷️ CONVERT IMAGES + CREDIT + LOGO — use what's ready on Drive, ║
# ║       download only what's missing                                ║
# ║                                                                    ║
# ║  Converts photos from the image sheet (Configuration cell) into   ║
# ║  still video segments, DURACAO_CLIPE seconds each, pulling the    ║
# ║  next UNUSED row in sequence (anti-repeat, same logic as video     ║
# ║  No verse matching in this notebook -- see                       ║
# ║  video-base-imagem-versiculo.ipynb for that.                      ║
# ║                                                                    ║
# ║  Checkpoint is isolated per video (checkpoint_[NAME].json).       ║
# ╚══════════════════════════════════════════════════════════════════╝

clipes = pipeline.baixar_clipes_imagem()

print(f"\n✅ {len(clipes)} image segment(s) ready in clipes_cortados/ ({DURACAO_CLIPE}s each)")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  💾 SAVE CLIPS TO DRIVE (optional)                               ║
# ║                                                                    ║
# ║  Copies this video's cut/credited clips to the shared pool         ║
# ║  (assets/clipes/), so future videos can reuse them instead of      ║
# ║  downloading from the sheet again.                                 ║
# ║                                                                    ║
# ║  TOTALLY OPTIONAL — safe to skip.                                 ║
# ╚══════════════════════════════════════════════════════════════════╝

n_salvos = pipeline.salvar_clipes_no_drive(clipes)
print(f"💾 {n_salvos} new clip(s) saved to {config.pasta_assets_clipes}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎬 VIDEO BASE — concatenate + narration + background music      ║
# ║  Clips arrive here already cut and credited (previous cell);      ║
# ║  this step just joins everything and mixes in the audio.          ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from ffmpeg_utils import obter_duracao

video_base = pipeline.criar_video_base(clipes)

if video_base and video_base.exists():
    duracao_final  = obter_duracao(video_base)
    duracao_audio  = obter_duracao(Path(config.NOME_AUDIO))
    print(f"✅ VIDEO BASE: {video_base.name} ({video_base.stat().st_size/1_048_576:.1f} MB, {duracao_final:.1f}s)")
    if abs(duracao_final - duracao_audio) > 1.0:
        print(f"⚠️  Warning: video ({duracao_final:.1f}s) doesn't match the audio ({duracao_audio:.1f}s) — please check.")
    else:
        print(f"✅ Duration matches the audio ({duracao_audio:.1f}s)")
else:
    print("⚠️  Video base was not generated — check the logs above.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  👀 PREVIEW VIDEO BASE                                           ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from IPython.display import Video, display

video_base = Path(config.NOME_VIDEO_BASE)
if video_base.exists():
    print(f"🎬 {video_base.name}  ({video_base.stat().st_size/1_048_576:.1f} MB)")
    display(Video(str(video_base), embed=True, width=800))
else:
    print(f"❌ Video base not found: {video_base}")
    print("   Run the clip-cutting and video-base cells first.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📥 DOWNLOAD — VIDEO BASE                                        ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from google.colab import files

video_base = Path(config.NOME_VIDEO_BASE)
if video_base.exists():
    print(f"📥 Downloading {video_base.name} ({video_base.stat().st_size/1_048_576:.1f} MB)...")
    files.download(str(video_base))
else:
    print(f"❌ Video base not found: {video_base}")


### 🔧 Utilities

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🧹 SELECTIVE CLEANUP                                            ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import shutil

def limpeza_seletiva():
    print("=" * 60)
    print("🧹 SELECTIVE CLEANUP")
    print("=" * 60)
    print("  1 - 🎵 Audio (.wav, .mp3)")
    print("  2 - 🎬 Generated videos (base, clips)")
    print("  3 - 📌 Checkpoint")
    print("  4 - 📁 Temp folders (clipes_cortados/, temp_raw/)")
    print("  5 - 🔥 EVERYTHING (1-4)")
    print("  0 - Cancel")
    escolha = input("\nEnter numbers separated by commas: ").strip()
    if escolha == '0':
        return
    opcoes = [int(x.strip()) for x in escolha.split(',')]
    cont = 0

    if 1 in opcoes or 5 in opcoes:
        for f in Path('.').glob('*_audio.wav'):
            f.unlink(); cont += 1; print(f"   🗑️ {f.name}")

    if 2 in opcoes or 5 in opcoes:
        for pattern in ['*_video_base.mp4', 'video_com_audio.mp4', 'video_sem_audio.mp4']:
            for f in Path('.').glob(pattern):
                f.unlink(); cont += 1; print(f"   🗑️ {f.name}")

    if 3 in opcoes or 5 in opcoes:
        for cp_file in Path('.').glob('checkpoint*.json'):
            cp_file.unlink(); cont += 1; print(f"   🗑️ {cp_file.name}")

    if 4 in opcoes or 5 in opcoes:
        for pasta in ['clipes_cortados', 'temp_raw', '__pycache__']:
            p = Path(pasta)
            if p.exists():
                shutil.rmtree(p); cont += 1; print(f"   🗑️ {pasta}/")

    print(f"\n✅ {cont} item(s) removed")

limpeza_seletiva()
